In [1]:
# ============================================================
# Notebook 03: Preprocessing Pipeline
# AI Financial Risk Intelligence Platform
# ============================================================

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "german.data"

# Column names
column_names = [
    "status", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "existing_credits", "job",
    "people_liable", "telephone", "foreign_worker", "target"
]

# Load data
df = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

# Convert target
df["target"] = df["target"].map({1: 0, 2: 1})

print("Dataset loaded successfully")
print(df.shape)

Dataset loaded successfully
(1000, 21)


In [2]:
# ============================================================
# Separate Features and Target
# ============================================================

X = df.drop(columns="target")
y = df["target"]

print("Feature Shape:", X.shape)
print("Target Shape :", y.shape)

X.head()

Feature Shape: (1000, 20)
Target Shape : (1000,)


,status,duration,credit_history,purpose,credit_amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


In [3]:
# ============================================================
# Train/Test Split
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain Target Distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest Target Distribution:")
print(y_test.value_counts(normalize=True).round(3))

X_train: (800, 20)
X_test : (200, 20)

Train Target Distribution:
target
0    0.7
1    0.3
Name: proportion, dtype: float64

Test Target Distribution:
target
0    0.7
1    0.3
Name: proportion, dtype: float64


In [4]:
# ============================================================
# Identify Feature Types
# ============================================================

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numerical_cols = X_train.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical Features:", len(numerical_cols))
print(numerical_cols)

Categorical Features: 13
['status', 'credit_history', 'purpose', 'savings', 'employment_duration', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

Numerical Features: 7
['duration', 'credit_amount', 'installment_rate', 'present_residence', 'age', 'existing_credits', 'people_liable']


In [5]:
# ============================================================
# Build the ColumnTransformer
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['duration', 'credit_amount',
                                  'installment_rate', 'present_residence',
                                  'age', 'existing_credits', 'people_liable']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['status', 'credit_history', 'purpose',
                                  'savings', 'employment_duration',
                                  'personal_status_sex', 'other_debtors',
                                  'property', 'other_installment_plans',
                                  'housing', 'job', 'telephone',
                                  'foreign_worker'])])


In [6]:
# ============================================================
# Transform the Training Data
# ============================================================

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Original X_train shape:", X_train.shape)
print("Transformed X_train shape:", X_train_transformed.shape)

print("\nOriginal X_test shape:", X_test.shape)
print("Transformed X_test shape:", X_test_transformed.shape)

Original X_train shape: (800, 20)
Transformed X_train shape: (800, 61)

Original X_test shape: (200, 20)
Transformed X_test shape: (200, 61)


In [7]:
# ============================================================
# Inspect Generated Feature Names
# ============================================================

feature_names = preprocessor.get_feature_names_out()

print("Total Features:", len(feature_names))
print()

for name in feature_names[:25]:
    print(name)

Total Features: 61

num__duration
num__credit_amount
num__installment_rate
num__present_residence
num__age
num__existing_credits
num__people_liable
cat__status_A11
cat__status_A12
cat__status_A13
cat__status_A14
cat__credit_history_A30
cat__credit_history_A31
cat__credit_history_A32
cat__credit_history_A33
cat__credit_history_A34
cat__purpose_A40
cat__purpose_A41
cat__purpose_A410
cat__purpose_A42
cat__purpose_A43
cat__purpose_A44
cat__purpose_A45
cat__purpose_A46
cat__purpose_A48


In [8]:
# ============================================================
# Build the ML Pipeline
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['duration', 'credit_amount',
                                                   'installment_rate',
                                                   'present_residence', 'age',
                                                   'existing_credits',
                                                   'people_liable']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['status', 'credit_history',
                                                   'purpose', 'savings',
                                                   'employment_duration',
                                                   'personal_status_sex',
                                                   'other_debtors', 'prope